In [65]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import math
from torch.utils.data import Dataset, DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

class SequenceLogDataset(Dataset):
    def __init__(self, df, pipeline, seq_len=10, is_train=True):
        """
        Loads raw CSV, applies the pipeline, and serves sliding windows for Transformers.
        """
        super().__init__()
        self.seq_len = seq_len
        
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df = df.sort_values('timestamp').reset_index(drop=True)
            df['timestamp'] = df['timestamp'].astype(str)
        else:
            print("WARNING: No timestamp column found. Sequences may be meaningless!")
        
        # 3. Separate features and labels
        if 'label' in df.columns:
            y_raw = df['label'].values
            X_raw = df.drop('label', axis=1)
        else:
            y_raw = np.zeros(len(df))
            X_raw = df

        print(f"Processing {'training' if is_train else 'testing'} data...")
        if is_train:
            X_processed = pipeline.fit_transform(X_raw)
        else:
            X_processed = pipeline.transform(X_raw)
        
        # Massive continuous pytorch tensors
        self.features = torch.tensor(X_processed, dtype=torch.float32)
        self.labels = torch.tensor(y_raw, dtype=torch.float32)
        
        print(f"Dataset ready. Total rows: {len(self.features)}. Sequence length: {self.seq_len}.")

    def __len__(self):
        return len(self.features) - self.seq_len + 1

    def __getitem__(self, idx):
        x_window = self.features[idx : idx + self.seq_len]
        window_labels = self.labels[idx : idx + self.seq_len]
        
        # If ANY log in this sequence is an attack, the sequence is an anomaly
        is_anomaly = 1.0 if torch.any(window_labels > 0) else 0.0
        return x_window, torch.tensor([is_anomaly], dtype=torch.float32)
    
    

In [66]:
from data.transformers import (
    ContentLengthTransformer,
    DateTransformer,
    URLFeatureExtractor,
)

EXPECTED_INPUT_COLUMNS = [
    "timestamp",
    "user",
    "client_ip",  # parsed but currently unused (IPTransformer commented out)
    "method",
    "url",
    "content_length",
]

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def build_preprocessor() -> Pipeline: # Notice we return a Pipeline now
    """Return the fitted-on-demand preprocessor with scaling."""
    
    content_pipeline = Pipeline(
        [("extractor", ContentLengthTransformer())]
    )

    # 1. The Feature Extractor
    preprocessor = ColumnTransformer(
        transformers=[
            ("time", DateTransformer(), ["timestamp"]),
            ("url_text", URLFeatureExtractor(), ["url"]),
            # Fix: sparse_output=False ensures PyTorch can convert it to a tensor
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["method", "user"]),
            ("num", content_pipeline, ["content_length"]),
        ]
    )
    
    # 2. The Final Pipeline (Extraction + Scaling)
    full_pipeline = Pipeline(steps=[
        ('features', preprocessor),
        ('scaler', StandardScaler()) 
    ])

    return full_pipeline

# 1. Initialize your shared preprocessing pipeline
preprocessor_pipeline = build_preprocessor()

# 2. Create the Datasets (Note: seq_len=10)
train_dataset = SequenceLogDataset(
    df=pd.read_csv('data/csic-2010/train_dataset.csv'), 
    pipeline=preprocessor_pipeline, 
    seq_len=10,
    is_train=True
)

test_dataset = SequenceLogDataset(
    df=pd.read_csv('data/csic-2010/test_dataset.csv'), 
    pipeline=preprocessor_pipeline, 
    seq_len=10,
    is_train=False
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# 4. Check the shapes to ensure the Transformer will accept it
sample_x, sample_y = next(iter(train_loader))

print(f"Batch X shape: {sample_x.shape}") # Expected: [128, 10, num_features]
print(f"Batch y shape: {sample_y.shape}") # Expected: [128, 1]

# 5. Initialize the Sequence Transformer
input_dim = sample_x.shape[2]
# m = LogSequenceTransformer(feature_dim=input_dim, seq_len=10)

Processing training data...
Dataset ready. Total rows: 48852. Sequence length: 10.
Processing testing data...
Dataset ready. Total rows: 12213. Sequence length: 10.
Batch X shape: torch.Size([128, 10, 67])
Batch y shape: torch.Size([128, 1])


/opt/anaconda3/envs/deepl/lib/python3.10/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [60]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) # Shape: (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class LogSequenceTransformer(nn.Module):
    def __init__(self, feature_dim, d_model=96, nhead=8, num_layers=4, seq_len=10):
        super().__init__()
        
        self.input_projection = nn.Linear(feature_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, 
                                                    dim_feedforward=d_model*4, 
                                                    dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.output_projection = nn.Linear(d_model, feature_dim)

    def forward(self, src):
        # src shape: [Batch, Seq_Len, Feature_Dim]
        x = self.input_projection(src)
        x = self.pos_encoder(x)
        
        memory = self.transformer_encoder(x) 
        reconstructed = self.output_projection(memory)
        return reconstructed

In [61]:
def train_transformer(epochs, feature_dim, seq_len=10):
    train_dataset = train_dataset = SequenceLogDataset(csv_file_path='data/csic-2010/train_dataset.csv', 
                                                    pipeline=preprocessor_pipeline, 
                                                    seq_len=seq_len,
                                                    is_train=True)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

    model = LogSequenceTransformer(feature_dim=feature_dim, seq_len=seq_len)
    
    # We use MSE because of reconstruction
    loss_fn = nn.MSELoss() 
    optim = torch.optim.Adam(model.parameters(), lr=1e-4)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for epoch in range(epochs):
        model.train()
        print(f"Starting Epoch: {epoch}")
        
        for i, (x_batch, y_batch) in enumerate(train_loader):
            # FILTER: Only train on purely normal sequences (y_batch == 0)
            normal_mask = (y_batch == 0).squeeze()
            
            if not normal_mask.any():
                continue 
            
            x_normal = x_batch[normal_mask].to(device)
            reconstructed = model(x_normal)
            
            # Compute loss between original sequence and reconstructed sequence
            loss = loss_fn(reconstructed, x_normal)

            optim.zero_grad()
            loss.backward()
            optim.step()
            
            if i % 25 == 0:
                print(f"\tStep {i}/{len(train_loader)}, Reconstruction Loss: {loss.item():.6f}")
                
    return model


m_transformer = train_transformer(epochs=15, feature_dim=67, seq_len=10)

Processing training data...
Dataset ready. Total rows: 48852. Sequence length: 10.
Starting Epoch: 0
	Step 0/382, Reconstruction Loss: 1.230531
	Step 25/382, Reconstruction Loss: 0.879305
	Step 50/382, Reconstruction Loss: 0.844016
	Step 75/382, Reconstruction Loss: 0.792807
	Step 100/382, Reconstruction Loss: 0.733828
	Step 125/382, Reconstruction Loss: 0.652673
	Step 150/382, Reconstruction Loss: 0.549730
	Step 175/382, Reconstruction Loss: 0.471573
	Step 200/382, Reconstruction Loss: 0.411579
	Step 225/382, Reconstruction Loss: 0.359277
	Step 250/382, Reconstruction Loss: 0.318312
	Step 275/382, Reconstruction Loss: 0.292125
	Step 300/382, Reconstruction Loss: 0.264231
	Step 325/382, Reconstruction Loss: 0.244271
	Step 350/382, Reconstruction Loss: 0.213973
	Step 375/382, Reconstruction Loss: 0.196349
Starting Epoch: 1
	Step 0/382, Reconstruction Loss: 0.195838
	Step 25/382, Reconstruction Loss: 0.176628
	Step 50/382, Reconstruction Loss: 0.157189
	Step 75/382, Reconstruction Loss: 

In [62]:
# Save the parameters to a file (conventionally .pth or .pt)
torch.save(m_transformer.state_dict(), 'big_transformer_weights.pth')


In [64]:
@torch.no_grad()
def evaluate_sequence_model(model, seq_len=10, threshold=None):
    test_dataset = SequenceLogDataset(csv_file_path='data/csic-2010/test_dataset.csv', 
                                    pipeline=preprocessor_pipeline, 
                                    seq_len=seq_len,
                                    is_train=False
                                )
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    device = next(model.parameters()).device
    model.eval()
    
    all_errors = []
    all_labels = []
    
    loss_fn = nn.MSELoss(reduction='none')
    
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        
        reconstructed = model(x_batch)
        window_errors = loss_fn(reconstructed, x_batch).mean(dim=[1, 2])
        
        all_errors.append(window_errors.cpu().numpy())
        all_labels.append(y_batch.numpy()) # y_batch is already on CPU
        
    errors = np.concatenate(all_errors)
    labels = np.concatenate(all_labels)
    
    if threshold is None:
        threshold = np.percentile(errors, 50)
        print(f"Dynamic Anomaly Threshold set to: {threshold:.6f}")
        
    # Predict 1 (Anomaly) if error > threshold
    predictions = (errors > threshold).astype(int)
    
    from sklearn.metrics import classification_report, roc_auc_score
    print("\n--- Sequence Detection Results ---")
    print(classification_report(labels, predictions, target_names=["Normal", "Anomaly"]))
    print(f"ROC-AUC Score: {roc_auc_score(labels, errors):.4f}")
    
    return predictions, labels, errors

# Usage:
preds, true_labels, mse_scores = evaluate_sequence_model(m_transformer, seq_len=10)

Processing testing data...


/opt/anaconda3/envs/deepl/lib/python3.10/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Dataset ready. Total rows: 12213. Sequence length: 10.
Dynamic Anomaly Threshold set to: 0.027648

--- Sequence Detection Results ---
              precision    recall  f1-score   support

      Normal       0.63      0.85      0.72      4530
     Anomaly       0.89      0.71      0.79      7674

    accuracy                           0.76     12204
   macro avg       0.76      0.78      0.76     12204
weighted avg       0.79      0.76      0.76     12204

ROC-AUC Score: 0.8381
